In [1]:
#required libraries
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [2]:
#Load the dataset
df_churn = pd.read_csv('churn-bigml-80.csv')

In [3]:
# 2. Preprocess the Data 
# Convert the target variable 'Churn' (True/False) to 1 and 0
df_churn['Churn'] = df_churn['Churn'].astype(int)

# Convert other categorical Yes/No columns to 1 and 0
df_churn['International plan'] = df_churn['International plan'].map({'Yes': 1, 'No': 0})
df_churn['Voice mail plan'] = df_churn['Voice mail plan'].map({'Yes': 1, 'No': 0})

# For 'State', we will use pandas get_dummies to One-Hot Encode it
df_churn = pd.get_dummies(df_churn, columns=['State'], drop_first=True)

# Separate features (X) and target (y)
X = df_churn.drop('Churn', axis=1)
y = df_churn['Churn']

In [4]:
# Split the dataset into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature Scaling: Standardize the data so all features have equal weight
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [5]:
print("--- Training Models ---")

# Model A: Logistic Regression
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_scaled, y_train)
y_pred_log = log_model.predict(X_test_scaled)

# Model B: Random Forest Classifier
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_scaled, y_train)
y_pred_rf = rf_model.predict(X_test_scaled)

--- Training Models ---


In [6]:
# Evaluate models using accuracy, precision, recall, and F1-score
def evaluate_model(name, y_true, y_pred):
    print(f"\nEvaluation for {name}:")
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"Recall:    {recall_score(y_true, y_pred):.4f}")
    print(f"F1-score:  {f1_score(y_true, y_pred):.4f}")

evaluate_model("Logistic Regression", y_test, y_pred_log)
evaluate_model("Random Forest", y_test, y_pred_rf)


Evaluation for Logistic Regression:
Accuracy:  0.8408
Precision: 0.4318
Recall:    0.2405
F1-score:  0.3089

Evaluation for Random Forest:
Accuracy:  0.9270
Precision: 0.9762
Recall:    0.5190
F1-score:  0.6777


In [7]:
# Grid Search (on Random Forest)
print("\n--- Performing Grid Search on Random Forest ---")
# the parameters we want to test
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [None, 10, 20]
}

# Grid search using 3-fold cross-validation
grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=3, scoring='accuracy')
grid_search.fit(X_train_scaled, y_train)

print(f"Best Parameters found by Grid Search: {grid_search.best_params_}")
print(f"Best Grid Search Accuracy: {grid_search.best_score_:.4f}")


--- Performing Grid Search on Random Forest ---
Best Parameters found by Grid Search: {'max_depth': 20, 'n_estimators': 50}
Best Grid Search Accuracy: 0.9334
